### BYTE PAIR ENCODING

<div class="alert alert-block alert-success">

We implemented a simple tokenization scheme in the previous sections for illustration
purposes. 

This section covers a more sophisticated tokenization scheme based on a concept
called byte pair encoding (BPE). 

The BPE tokenizer covered in this section was used to train
LLMs such as GPT-2, GPT-3, and the original model used in ChatGPT.</div>

<div class="alert alert-block alert-warning">

Since implementing BPE can be relatively complicated, we will use an existing Python
open-source library called tiktoken (https://github.com/openai/tiktoken). 

This library implements
the BPE algorithm very efficiently based on source code in Rust.
</div>

**BPE Tokenizer**

In [ ]:
! pip3 install tiktoken

In [2]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.11.0


<div class="alert alert-block alert-success">
Once installed, we can instantiate the BPE tokenizer from tiktoken as follows:</div>

In [3]:
tokenizer = tiktoken.get_encoding("gpt2")

If we were to use word level tokenizer, English Language has about 170,000 to 200,000 words. So, the vocabulary size would have been this number.

![Word Level Tokens](../assets/img/word-lvl-token.png)

<div class="alert alert-block alert-success">
The usage of this tokenizer is similar to SimpleTokenizerV2 we implemented previously via
an encode method:</div>



In [4]:
text = (
    "Hello, do you like chai? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 442, 1872, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


<div class="alert alert-block alert-info">
    
The code above prints the following token IDs:

</div>

<div class="alert alert-block alert-success">
We can then convert the token IDs back into text using the decode method, similar to our
SimpleTokenizerV2 earlier:</div>

In [5]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like chai? <|endoftext|> In the sunlit terracesof someunknownPlace.


<div class="alert alert-block alert-warning">

We can make two noteworthy observations based on the token IDs and decoded text
above. 

First, the <|endoftext|> token is assigned a relatively large token ID, namely,
50256. 

In fact, the BPE tokenizer, which was used to train models such as GPT-2, GPT-3,
and the original model used in ChatGPT, has a total vocabulary size of 50,257, with
<|endoftext|> being assigned the largest token ID.
    


</div>

<div class="alert alert-block alert-warning">

Second, the BPE tokenizer above encodes and decodes unknown words, such as
"someunknownPlace" correctly. 

The BPE tokenizer can handle any unknown word. How does
it achieve this without using <|unk|> tokens?
    


</div>

<div class="alert alert-block alert-warning">

The algorithm underlying BPE breaks down words that aren't in its predefined vocabulary
into smaller subword units or even individual characters.

The enables it to handle out-ofvocabulary words. 

So, thanks to the BPE algorithm, if the tokenizer encounters an
unfamiliar word during tokenization, it can represent it as a sequence of subword tokens or
characters
    


</div>

**Let us take another simple example to illustrate how the BPE tokenizer deals with unknown tokens**

**Exercise**

In [6]:
integers = tokenizer.encode("Akwirw ier")
print(integers)

strings = tokenizer.decode(integers)
print(strings)

[33901, 86, 343, 86, 220, 959]
Akwirw ier


**Data sampling with sliding window**

In [7]:
from datasets import load_dataset

ds = load_dataset("elricwan/HarryPotter")

# Find and extract the Harry-Potter.txt file (the 8th file with all content)
harry_potter_full = None

for i in range(len(ds['train'])):
    if ds['train'][i]['filename'] == 'Harry-Potter.txt':
        harry_potter_full = ds['train'][i]['content']
        print(f"Found 'Harry-Potter.txt' at index {i}")
        print(f"Content length: {len(harry_potter_full)} characters")
        print(f"\nFirst 200 characters:\n{harry_potter_full[:50]}")
        break

#  Use this as raw_text
if harry_potter_full:
    raw_text = harry_potter_full
    print(f"\n✓ Successfully loaded Harry-Potter.txt into raw_text variable")
else:
    print("Harry-Potter.txt not found in dataset")

Found 'Harry-Potter.txt' at index 7
Content length: 6491209 characters

First 200 characters:
FOR JESSICA, WHO LOVES STORIES,

FOR ANNE, WHO LOV

✓ Successfully loaded Harry-Potter.txt into raw_text variable


In [8]:
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

1790693


In [9]:
enc_sample = enc_text[50:]

In [10]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [198, 464, 6387, 5338]
y:      [464, 6387, 5338, 406]


In [11]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[198] ----> 464
[198, 464] ----> 6387
[198, 464, 6387] ----> 5338
[198, 464, 6387, 5338] ----> 406


In [12]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))


 ----> The

The ---->  Boy

The Boy ---->  Who

The Boy Who ---->  L


**IMPLEMENTING A DATA LOADER**

In [13]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [14]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [15]:
import torch

print("PyTorch version:", torch.__version__)
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

PyTorch version: 2.7.1
[tensor([[13775,   449,  7597, 25241]]), tensor([[  449,  7597, 25241,    11]])]


In [16]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[  449,  7597, 25241,    11]]), tensor([[ 7597, 25241,    11, 19494]])]


In [17]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[13775,   449,  7597, 25241],
        [   11, 19494,   406,  8874],
        [ 1546, 46366, 11015,    11],
        [  198,   198, 13775,  3537],
        [12161,    11, 19494,   406],
        [ 8874,  1961, 44788,  5390],
        [   46,    26,   198,   198],
        [ 6981,  7473, 14766,    11]])

Targets:
 tensor([[  449,  7597, 25241,    11],
        [19494,   406,  8874,  1546],
        [46366, 11015,    11,   198],
        [  198, 13775,  3537, 12161],
        [   11, 19494,   406,  8874],
        [ 1961, 44788,  5390,    46],
        [   26,   198,   198,  6981],
        [ 7473, 14766,    11, 19494]])


**CREATE TOKEN EMBEDDINGS**

In [18]:
input_ids = torch.tensor([2, 3, 5, 1])

In [19]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [20]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [21]:
print(embedding_layer(torch.tensor([3])))


tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [22]:
print(embedding_layer(input_ids))


tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


**POSITIONAL EMBEDDINGS (ENCODING WORD POSITIONS)**

In [23]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [24]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [25]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[13775,   449,  7597, 25241],
        [   11, 19494,   406,  8874],
        [ 1546, 46366, 11015,    11],
        [  198,   198, 13775,  3537],
        [12161,    11, 19494,   406],
        [ 8874,  1961, 44788,  5390],
        [   46,    26,   198,   198],
        [ 6981,  7473, 14766,    11]])

Inputs shape:
 torch.Size([8, 4])


In [26]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [27]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

In [28]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [29]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])
